# RideBase 06 — V1 Regression Benchmark

Bu notebook RideBase Synthetic Dataset v1.2 üzerinde klasik regression modellerini karşılaştırarak sonraki servise kalan **gün** ve **kilometreyi** ayrı ayrı tahmin eder. Model taraması TRAIN → VALIDATION üzerinde yapılır; TEST yalnız seçilmiş final modellerin tarafsız değerlendirmesinde bir kez kullanılır.

### Ne yapıyoruz?
*Supervised learning*, geçmiş örneklerde hem girdilerin hem de doğru cevabın bulunduğu öğrenmedir. *Regression* ise cevabın gün veya kilometre gibi sayısal olduğu problemdir.

### Neden yapıyoruz?
V0 kural sisteminin üstüne, verideki çok sayıdaki mevcut ve geçmiş sinyali birlikte kullanan sade bir ML referansı koyuyoruz. *Baseline*, daha gelişmiş yaklaşımın gerçekten fayda sağlayıp sağlamadığını ölçtüğümüz başlangıç çizgisidir.

### Sonuç nasıl yorumlanmalı?
Sonuçlar yalnız sentetik v1.2 offline benchmarkıdır; production performansı iddiası değildir.


In [1]:
from pathlib import Path
import hashlib, inspect, json, platform, time, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn import __version__ as sklearn_version
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.linear_model import (BayesianRidge, ElasticNet, GammaRegressor, Lasso,
                                  LinearRegression, PassiveAggressiveRegressor,
                                  PoissonRegressor, Ridge, SGDRegressor, TweedieRegressor)
from sklearn.ensemble import (AdaBoostRegressor, BaggingRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.svm import LinearSVR
from lazypredict.Supervised import LazyRegressor, REGRESSORS
import lazypredict.Supervised as lazy_supervised
try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")

ROOT = find_project_root()
DATASET_ROOT = ROOT.parent / "ridebase_v1_2"
DERIVED = DATASET_ROOT / "derived_outputs"
MODELS = ROOT / "models"
OUTPUTS = ROOT / "outputs"
REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures" / "v1_regression"
for directory in [MODELS, OUTPUTS, TABLES, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)
SEED = 42
DATASET_VERSION = "1.2.0"


## 1. Dataset ve sürüm koruması

### Ne yapıyoruz?
Merkezi v1.2 yollarını kullanıyor, metadata sürümlerini ve gerçek split sayılarını kontrol ediyoruz.

### Neden yapıyoruz?
Eski v1/v1.1 dosyalarıyla sessizce farklı sonuç üretmeyi engelliyoruz. Bu hücre geçmezse çalışma durur.

### Sonuç nasıl yorumlanmalı?
`DATASET_VERSION_GUARD=PASS`, doğru ve beklenen veri sürümünün kullanıldığı anlamına gelir.


In [2]:
with open(DERIVED / "dataset_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)
dataset_info = metadata["dataset"]
if dataset_info.get("dataset_version") != DATASET_VERSION or dataset_info.get("generator_version") != DATASET_VERSION:
    raise RuntimeError("Yalnız authoritative RideBase v1.2.0 kullanılabilir; eski v1/v1.1 path reddedildi")

snapshots = pd.read_parquet(DERIVED / "ml_maintenance_snapshots.parquet")
targets = pd.read_parquet(DERIVED / "ml_next_service_targets.parquet")
split_manifest = pd.read_csv(DERIVED / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)
manifest = pd.read_parquet(OUTPUTS / "ml_modeling_manifest.parquet")

expected_split = {"TRAIN": 27428, "VALIDATION": 6399, "TEST": 7691}
actual_split = manifest["split"].value_counts().to_dict()
if len(snapshots) != 41518 or len(targets) != 41518 or len(manifest) != 41518 or actual_split != expected_split:
    raise RuntimeError(f"Dataset/split guard failed: {len(snapshots)}, {len(targets)}, {len(manifest)}, {actual_split}")
for frame, name in [(snapshots,"snapshots"),(targets,"targets"),(manifest,"manifest")]:
    if frame["snapshot_id"].duplicated().any():
        raise RuntimeError(f"Duplicate snapshot_id: {name}")
print("DATASET_VERSION_GUARD=PASS", DATASET_VERSION, actual_split)


DATASET_VERSION_GUARD=PASS 1.2.0 {'TRAIN': 27428, 'TEST': 7691, 'VALIDATION': 6399}


## 2. 05 preprocessing artifactı ve observed-only sözleşme

### Ne yapıyoruz?
05'in TRAIN üzerinde fit ettiği ön işlemciyi ve model manifestini aynen kullanıyoruz; yeni bir imputer, scaler veya encoder öğrenmiyoruz.

### Neden yapıyoruz?
Target tablolarını X'e katmamak leakage'i önler. V1 yalnız gerçekten sonraki servisi görülmüş satırları kullanır. Örneğin bir araç 120 gündür dönmediyse gerçek sonraki servis günü bilinmez; bu kayda 120, 0 veya -1 hedefi vermek yanlış olur.

### Sonuç nasıl yorumlanmalı?
Observed sayıları gün hedefi için 20.679 / 1.932 / 2.622 olmalıdır. Kilometrede ayrıca geçerli ölçüm kontrolü uygulanır; geçersiz gözlem saklanmaz, açıkça raporlanır.


In [3]:
preprocessor_path = MODELS / "ml_preprocessor_v1_2.joblib"
preprocessor = joblib.load(preprocessor_path)
raw_features = list(preprocessor.feature_names_in_)
encoded_features = np.asarray(preprocessor.get_feature_names_out(), dtype=str)
if len(raw_features) != 127 or len(encoded_features) != 277:
    raise RuntimeError(f"05 feature contract mismatch: raw={len(raw_features)}, encoded={len(encoded_features)}")
for forbidden in ["target", "future", "next_service", "next_task"]:
    if any(forbidden in x.lower() for x in raw_features):
        raise RuntimeError(f"Leakage feature found: {forbidden}")

joined = (manifest.merge(targets[["snapshot_id","target_event_observed","days_to_next_service","km_to_next_service","target_km_valid"]],
                         on="snapshot_id", how="left", validate="one_to_one", suffixes=("_manifest","_target"))
                  .merge(snapshots, on=["snapshot_id","motorcycle_id"], how="left", validate="one_to_one"))
joined = joined.set_index("snapshot_id", drop=False)
days_col, km_col = "days_to_next_service_target", "km_to_next_service_target"
days_mask = joined["v1_regression_eligible"].astype(bool) & joined[days_col].notna() & np.isfinite(joined[days_col]) & (joined[days_col] >= 0)
km_mask = days_mask & joined[km_col].notna() & np.isfinite(joined[km_col]) & (joined[km_col] >= 0) & (joined["target_km_valid_target"] == 1)

observed_counts = joined.loc[days_mask].groupby("split").size().to_dict()
expected_observed = {"TRAIN":20679,"VALIDATION":1932,"TEST":2622}
if observed_counts != expected_observed:
    raise RuntimeError(f"Observed mask mismatch: {observed_counts}")

validity_rows = []
for target_name, column, mask in [("DAYS",days_col,days_mask),("KM",km_col,km_mask)]:
    for split in expected_split:
        eligible = (joined["split"] == split) & joined["v1_regression_eligible"].astype(bool)
        s = joined.loc[eligible, column]
        validity_rows.append({"target":target_name,"split":split,"eligible_rows":int(eligible.sum()),
                              "valid_rows":int((mask & (joined["split"]==split)).sum()),
                              "null":int(s.isna().sum()),"negative":int((s<0).sum()),"zero":int((s==0).sum()),
                              "inf":int(np.isinf(s.dropna()).sum()),"min":s.min(),"max":s.max()})
target_validity = pd.DataFrame(validity_rows)
target_validity.to_csv(TABLES / "v1_regression_target_validity.csv", index=False, encoding="utf-8-sig")
display(target_validity)


,target,split,eligible_rows,valid_rows,null,negative,zero,inf,min,max
0,DAYS,TRAIN,20679,20679,0,0,0,0,0.000694,1343.889583
1,DAYS,VALIDATION,1932,1932,0,0,0,0,0.000694,183.236806
2,DAYS,TEST,2622,2622,0,0,0,0,0.000694,204.846528
3,KM,TRAIN,20679,20678,0,1,18,0,-1.000000,63341.000000
4,KM,VALIDATION,1932,1931,0,1,5,0,-1.000000,18912.000000
5,KM,TEST,2622,2622,0,0,1,0,0.000000,20038.000000


## 3. Feature matrisleri ve sparse/dense kararı

### Ne yapıyoruz?
05 ön işlemcisiyle tüm splitleri dönüştürüyor, sonra yalnız hedefe uygun indeksleri alıyoruz. Sparse matrisin dense kopyasını oluşturmadan önce `satır × sütun × dtype byte` maliyetini hesaplıyoruz.

### Neden yapıyoruz?
LazyPredict içindeki bazı klasik modeller sparse girdi kabul etmez. Bu veri boyutunda float32 dense kopya güvenliyse tarama için kullanılır; 05'in feature contractı değişmez.

### Sonuç nasıl yorumlanmalı?
NaN/inf yoksa ve tüm splitlerde 277 sütun korunuyorsa feature matrisi modellemeye hazırdır.


In [4]:
X_all_sparse = preprocessor.transform(joined[raw_features])
if not sparse.issparse(X_all_sparse):
    raise RuntimeError("05 artifact sparse output contract failed")
if not np.isfinite(X_all_sparse.data).all() or X_all_sparse.shape != (41518,277):
    raise RuntimeError("Transformed feature QA failed")

split_pos = {s: np.flatnonzero(joined["split"].eq(s).to_numpy()) for s in expected_split}
days_pos = {s: np.flatnonzero((joined["split"].eq(s) & days_mask).to_numpy()) for s in expected_split}
km_pos = {s: np.flatnonzero((joined["split"].eq(s) & km_mask).to_numpy()) for s in expected_split}
dense_bytes = (len(days_pos["TRAIN"])+len(days_pos["VALIDATION"]))*len(encoded_features)*np.dtype("float32").itemsize
print(f"LazyPredict dense RAM estimate: {dense_bytes/1024**2:.2f} MiB")
if dense_bytes > 512*1024**2:
    raise MemoryError("Dense tarama kopyası 512 MiB güvenlik sınırını aşıyor")

Xd = {s: X_all_sparse[days_pos[s]].toarray().astype("float32", copy=False) for s in expected_split}
Xk = {s: X_all_sparse[km_pos[s]].toarray().astype("float32", copy=False) for s in expected_split}
yd = {s: joined.iloc[days_pos[s]][days_col].to_numpy(float) for s in expected_split}
yk = {s: joined.iloc[km_pos[s]][km_col].to_numpy(float) for s in expected_split}
matrix_qa = pd.DataFrame([
    {"check":"same_encoded_dimension","status":"PASS" if all(x.shape[1]==277 for x in [*Xd.values(),*Xk.values()]) else "FAIL"},
    {"check":"no_nan_inf","status":"PASS" if all(np.isfinite(x).all() for x in [*Xd.values(),*Xk.values(),*yd.values(),*yk.values()]) else "FAIL"},
    {"check":"observed_only","status":"PASS"},
])
if not matrix_qa["status"].eq("PASS").all(): raise RuntimeError("Feature matrix QA failed")
display(matrix_qa)


LazyPredict dense RAM estimate: 23.89 MiB


,check,status
0,same_encoded_dimension,PASS
1,no_nan_inf,PASS
2,observed_only,PASS


## 4. Basit baseline'lar

### Ne yapıyoruz?
Global median, herkese TRAIN hedef medyanını söyler. Segment median ise yalnız TRAIN'de en az 50 örneği bulunan `category + usage_type` gruplarının medyanını kullanır; bilinmeyen gruplarda global medyana döner.

### Neden yapıyoruz?
Böylece “hiç feature kullanmasaydık?” ve “yalnız kaba bir segment bilgisi kullansaydık?” sorularını leakage-free biçimde cevaplarız.

### Sonuç nasıl yorumlanmalı?
ML modelinin yalnız V0'ı değil, bu basit istatistiksel referansları da geçmesi beklenir.


In [5]:
def metrics(y, pred, target):
    pred = np.asarray(pred, float); y = np.asarray(y, float); ae=np.abs(pred-y)
    out={"mae":mean_absolute_error(y,pred),"median_ae":median_absolute_error(y,pred),
         "rmse":mean_squared_error(y,pred)**0.5,"r2":r2_score(y,pred),"bias":float(np.mean(pred-y)),
         "p90_ae":float(np.quantile(ae,.90))}
    tolerances=[30,60,90] if target=="DAYS" else [1000,2000,5000]
    out.update({f"within_{t}":float(np.mean(ae<=t)) for t in tolerances})
    return out

def segment_predictions(target, positions):
    ycol = days_col if target=="DAYS" else km_col
    mask = days_mask if target=="DAYS" else km_mask
    train = joined.loc[mask & joined["split"].eq("TRAIN"), ["category","usage_type",ycol]].copy()
    global_median=float(train[ycol].median())
    stats=train.groupby(["category","usage_type"])[ycol].agg(["median","size"])
    stats=stats[stats["size"]>=50]["median"]
    rows=joined.iloc[positions][["category","usage_type"]]
    idx=pd.MultiIndex.from_frame(rows)
    return np.array([stats.get(k,global_median) for k in idx],float),global_median

baseline_preds={}
for target,yset,positions in [("DAYS",yd,days_pos),("KM",yk,km_pos)]:
    global_med=float(np.median(yset["TRAIN"]))
    baseline_preds[(target,"Global Median")]=np.full(len(yset["TEST"]),global_med)
    baseline_preds[(target,"Segment Median")],_=segment_predictions(target,positions["TEST"])
    print(target,"TRAIN global median",global_med)


DAYS TRAIN global median 111.95277777777778
KM TRAIN global median 5267.5


## 5. LazyPredict ile TRAIN → VALIDATION taraması

### Ne yapıyoruz?
LazyPredict birçok klasik estimatorı aynı TRAIN/VALIDATION sözleşmesiyle hızlıca tarar. Validation, model seçeneklerini TEST'e dokunmadan karşılaştırdığımız ara sınavdır.

### Neden TEST üzerinde model seçilmez?
TEST'e bakarak seçim yapmak, final sınavının cevaplarını önceden görmek gibidir ve ölçümü iyimserleştirir. LazyPredict final model değildir: yalnız aday keşfeder; seçimden sonra gerçek sklearn estimator sınıfı ayrıca TRAIN üzerinde fit edilir.

### Regularization ve overfitting
*Regularization*, modelin aşırı karmaşık çözümlerini cezalandırarak ezberlemeyi azaltır. *Overfitting*, TRAIN çok iyiyken yeni veride performansın bozulmasıdır. Kernel tabanlı O(n²)/O(n³) ve saatler sürebilecek modeller runtime güvenliği için gerekçeli biçimde atlanır.

### Metrikler
MAE ortalama mutlak hatadır; Median AE tipik hatayı uç değerlerden daha az etkilenerek gösterir. RMSE büyük hataları daha fazla cezalandırır. R², modelin hedef değişkenliğini ne kadar açıkladığını ölçer; 1 iyi, 0 sabit ortalama düzeyi, negatif değer ise bundan kötü demektir.


In [6]:
SCREEN_CLASSES=[AdaBoostRegressor,BaggingRegressor,BayesianRidge,DecisionTreeRegressor,DummyRegressor,
                ElasticNet,ExtraTreeRegressor,ExtraTreesRegressor,GammaRegressor,GradientBoostingRegressor,
                HistGradientBoostingRegressor,Lasso,LinearRegression,LinearSVR,PassiveAggressiveRegressor,
                PoissonRegressor,Ridge,SGDRegressor,TweedieRegressor]
if XGBRegressor is not None: SCREEN_CLASSES.append(XGBRegressor)

def screen_target(target,X_train,X_val,y_train,y_val):
    original_numeric = lazy_supervised.numeric_transformer
    lazy_supervised.numeric_transformer = "passthrough"  # 05 zaten scaling/encoding yaptı
    try:
        lazy=LazyRegressor(verbose=0,ignore_warnings=True,custom_metric=None,predictions=True,
                           random_state=SEED,n_jobs=-1,regressors=SCREEN_CLASSES)
        lazy_scores,preds=lazy.fit(X_train,X_val,y_train,y_val)
    finally:
        lazy_supervised.numeric_transformer = original_numeric
    rows=[]
    for model in preds.columns:
        raw=preds[model].to_numpy(float); actionable=np.clip(raw,0,None)
        m=metrics(y_val,actionable,target); rawm=metrics(y_val,raw,target)
        row={"model":model,**m,"raw_mae":rawm["mae"],"raw_median_ae":rawm["median_ae"],
             "negative_predictions":int((raw<0).sum()),"clipping_mae_change":m["mae"]-rawm["mae"],
             "runtime":float(lazy_scores.loc[model,"Time Taken"]) if model in lazy_scores.index else np.nan,
             "status":"SUCCESS"}
        rows.append(row)
    ranking=pd.DataFrame(rows).sort_values(["mae","median_ae","model"]).reset_index(drop=True)
    ranking["rank"]=np.arange(1,len(ranking)+1)
    return ranking,lazy

days_ranking,days_lazy=screen_target("DAYS",Xd["TRAIN"],Xd["VALIDATION"],yd["TRAIN"],yd["VALIDATION"])
print("DAYS TOP 5"); display(days_ranking.head(5))


DAYS TOP 5


,model,mae,median_ae,rmse,r2,bias,p90_ae,within_30,within_60,within_90,raw_mae,raw_median_ae,negative_predictions,clipping_mae_change,runtime,status,rank
0,HistGradientBoostingRegressor,22.471394,16.723599,30.363077,0.283081,2.435359,50.106080,0.777433,0.933230,0.984990,22.471394,16.723599,0,0.000000,1.160517,SUCCESS,1
1,GradientBoostingRegressor,22.492793,15.888557,32.288004,0.189299,2.548562,52.016006,0.761905,0.924948,0.975673,22.500975,15.888557,1,-0.008182,24.822442,SUCCESS,2
2,LinearSVR,28.814702,19.289201,43.682824,-0.483883,7.909819,60.188556,0.679607,0.898551,0.950311,28.901599,19.361676,15,-0.086897,2.922686,SUCCESS,3
3,XGBRegressor,28.952914,20.344036,39.722684,-0.227031,-14.585824,67.292003,0.623706,0.859213,0.960663,29.130935,20.394902,23,-0.178021,0.806614,SUCCESS,4
4,PoissonRegressor,31.266741,25.083235,42.479579,-0.403262,18.593866,57.094410,0.601449,0.914079,0.959627,31.266741,25.083235,0,0.000000,1.514022,SUCCESS,5


In [7]:
km_ranking,km_lazy=screen_target("KM",Xk["TRAIN"],Xk["VALIDATION"],yk["TRAIN"],yk["VALIDATION"])
print("KM TOP 5"); display(km_ranking.head(5))

# LazyPredict taramasında yakınsamayan bir estimator iyi görüünebilir. Final estimatorın
# ayrı sklearn refit'inde aynı Validation MAE'yi üretmesi seçim ön koşuludur.
CLASS_MAP={c.__name__:c for c in SCREEN_CLASSES}
def build_estimator(name):
    cls=CLASS_MAP[name]; params={}
    sig=inspect.signature(cls)
    if "random_state" in sig.parameters: params["random_state"]=SEED
    if "n_jobs" in sig.parameters: params["n_jobs"]=-1
    if name=="XGBRegressor": params.update({"verbosity":0})
    return cls(**params)

def apply_refit_gate(ranking,target,Xtrain,Xval,ytrain,yval):
    ranking=ranking.copy(); ranking["screen_rank"]=ranking["rank"]
    ranking["selection_eligible"]=True; ranking["selection_note"]="Not checked; below first stable candidate"
    for idx in ranking.index:
        name=ranking.loc[idx,"model"]
        candidate=build_estimator(name).fit(Xtrain,ytrain)
        refit_mae=metrics(yval,np.clip(candidate.predict(Xval),0,None),target)["mae"]
        delta=abs(refit_mae-ranking.loc[idx,"mae"])/max(ranking.loc[idx,"mae"],1e-12)
        ranking.loc[idx,"refit_validation_mae"]=refit_mae
        ranking.loc[idx,"refit_relative_delta"]=delta
        if delta <= 0.005:
            ranking.loc[idx,"selection_note"]="PASS: separate sklearn refit reproduced Validation MAE within 0.5%"
            break
        ranking.loc[idx,"selection_eligible"]=False
        ranking.loc[idx,"selection_note"]="REJECTED: separate refit did not reproduce Validation MAE (likely non-convergence)"
    ranking=ranking.sort_values(["selection_eligible","mae","median_ae"],ascending=[False,True,True]).reset_index(drop=True)
    ranking["rank"]=np.arange(1,len(ranking)+1)
    return ranking

days_ranking=apply_refit_gate(days_ranking,"DAYS",Xd["TRAIN"],Xd["VALIDATION"],yd["TRAIN"],yd["VALIDATION"])
km_ranking=apply_refit_gate(km_ranking,"KM",Xk["TRAIN"],Xk["VALIDATION"],yk["TRAIN"],yk["VALIDATION"])
days_ranking.to_csv(TABLES/"v1_lazypredict_days_validation.csv",index=False,encoding="utf-8-sig")
km_ranking.to_csv(TABLES/"v1_lazypredict_km_validation.csv",index=False,encoding="utf-8-sig")
print("SELECTION-ELIGIBLE DAYS TOP 5"); display(days_ranking.head(5))
print("SELECTION-ELIGIBLE KM TOP 5"); display(km_ranking.head(5))

all_lazy_names=sorted({cls.__name__ for _,cls in REGRESSORS})
screened_names=sorted({cls.__name__ for cls in SCREEN_CLASSES})
runtime_rows=[]
for target,ranking in [("DAYS",days_ranking),("KM",km_ranking)]:
    success=set(ranking["model"])
    for model in all_lazy_names:
        if model in success:
            runtime_rows.append({"target":target,"model":model,"status":"SUCCESS","runtime":float(ranking.set_index("model").loc[model,"runtime"]),"skip_reason":""})
        elif model in screened_names:
            runtime_rows.append({"target":target,"model":model,"status":"FAILED","runtime":np.nan,"skip_reason":"Estimator bu hedefte prediction üretemedi"})
        else:
            reason="O(n²)/O(n³), kernel/neighbors/CV veya benchmarkı kilitleyecek maliyet" if model in {"GaussianProcessRegressor","KernelRidge","KNeighborsRegressor","MLPRegressor","NuSVR","SVR","QuantileRegressor","RANSACRegressor"} else "Runtime bütçesi ve aynı ailede daha güvenli temsilci mevcut"
            runtime_rows.append({"target":target,"model":model,"status":"SKIPPED","runtime":np.nan,"skip_reason":reason})
runtime_status=pd.DataFrame(runtime_rows)
runtime_status.to_csv(TABLES/"v1_lazypredict_runtime_status.csv",index=False,encoding="utf-8-sig")


KM TOP 5


,model,mae,median_ae,rmse,r2,bias,p90_ae,within_1000,within_2000,within_5000,raw_mae,raw_median_ae,negative_predictions,clipping_mae_change,runtime,status,rank
0,LinearSVR,1429.501050,516.266256,2441.615619,0.000269,-750.277165,4214.798817,0.663905,0.742620,0.931642,1429.501050,516.266256,0,0.000000,0.282496,SUCCESS,1
1,HistGradientBoostingRegressor,1482.750395,944.124910,2248.549506,0.152122,-358.129291,3515.028467,0.526670,0.772657,0.955981,1482.750395,944.124910,0,0.000000,1.027542,SUCCESS,2
2,GradientBoostingRegressor,1494.845354,950.195451,2238.258441,0.159865,-313.888734,3455.298940,0.522527,0.762817,0.953910,1496.981107,950.867899,7,-2.135752,20.641322,SUCCESS,3
3,PassiveAggressiveRegressor,1647.257236,1114.416305,2388.152389,0.043572,-333.700579,3913.840958,0.468151,0.723459,0.948213,1649.736054,1116.187973,6,-2.478818,0.232260,SUCCESS,4
4,BayesianRidge,1850.117601,1457.739990,2421.102236,0.016997,506.999527,3734.700195,0.327809,0.643190,0.955981,1853.369653,1459.515137,4,-3.252053,0.311092,SUCCESS,5


SELECTION-ELIGIBLE DAYS TOP 5


,model,mae,median_ae,rmse,r2,bias,p90_ae,within_30,within_60,within_90,raw_mae,raw_median_ae,negative_predictions,clipping_mae_change,runtime,status,rank,screen_rank,selection_eligible,selection_note,refit_validation_mae,refit_relative_delta
0,HistGradientBoostingRegressor,22.471394,16.723599,30.363077,0.283081,2.435359,50.106080,0.777433,0.933230,0.984990,22.471394,16.723599,0,0.000000,1.160517,SUCCESS,1,1,True,PASS: separate sklearn refit reproduced Valida...,22.471394,0.0
1,GradientBoostingRegressor,22.492793,15.888557,32.288004,0.189299,2.548562,52.016006,0.761905,0.924948,0.975673,22.500975,15.888557,1,-0.008182,24.822442,SUCCESS,2,2,True,Not checked; below first stable candidate,NaN,NaN
2,LinearSVR,28.814702,19.289201,43.682824,-0.483883,7.909819,60.188556,0.679607,0.898551,0.950311,28.901599,19.361676,15,-0.086897,2.922686,SUCCESS,3,3,True,Not checked; below first stable candidate,NaN,NaN
3,XGBRegressor,28.952914,20.344036,39.722684,-0.227031,-14.585824,67.292003,0.623706,0.859213,0.960663,29.130935,20.394902,23,-0.178021,0.806614,SUCCESS,4,4,True,Not checked; below first stable candidate,NaN,NaN
4,PoissonRegressor,31.266741,25.083235,42.479579,-0.403262,18.593866,57.094410,0.601449,0.914079,0.959627,31.266741,25.083235,0,0.000000,1.514022,SUCCESS,5,5,True,Not checked; below first stable candidate,NaN,NaN


SELECTION-ELIGIBLE KM TOP 5


,model,mae,median_ae,rmse,r2,bias,p90_ae,within_1000,within_2000,within_5000,raw_mae,raw_median_ae,negative_predictions,clipping_mae_change,runtime,status,rank,screen_rank,selection_eligible,selection_note,refit_validation_mae,refit_relative_delta
0,HistGradientBoostingRegressor,1482.750395,944.124910,2248.549506,0.152122,-358.129291,3515.028467,0.526670,0.772657,0.955981,1482.750395,944.124910,0,0.000000,1.027542,SUCCESS,1,2,True,PASS: separate sklearn refit reproduced Valida...,1482.750395,0.0
1,GradientBoostingRegressor,1494.845354,950.195451,2238.258441,0.159865,-313.888734,3455.298940,0.522527,0.762817,0.953910,1496.981107,950.867899,7,-2.135752,20.641322,SUCCESS,2,3,True,Not checked; below first stable candidate,NaN,NaN
2,PassiveAggressiveRegressor,1647.257236,1114.416305,2388.152389,0.043572,-333.700579,3913.840958,0.468151,0.723459,0.948213,1649.736054,1116.187973,6,-2.478818,0.232260,SUCCESS,3,4,True,Not checked; below first stable candidate,NaN,NaN
3,BayesianRidge,1850.117601,1457.739990,2421.102236,0.016997,506.999527,3734.700195,0.327809,0.643190,0.955981,1853.369653,1459.515137,4,-3.252053,0.311092,SUCCESS,4,5,True,Not checked; below first stable candidate,NaN,NaN
4,PoissonRegressor,1861.192585,1534.847255,2388.183903,0.043546,659.215596,3693.727508,0.282755,0.670119,0.962196,1861.192585,1534.847255,0,0.000000,1.280660,SUCCESS,5,6,True,Not checked; below first stable candidate,NaN,NaN


## 6. Validation seçimi ve final estimatorlar

### Ne yapıyoruz?
Ana seçim metriği ACTIONABLE Validation MAE, eşitlik bozucu Median AE'dir. Ürün açısından negatif gün/km anlamlı olmadığı için ana tahmin `max(raw, 0)`; model analizi için raw tahmin de saklanır.

### Neden yapıyoruz?
Validation sonucuna göre her hedef için yalnız bir model sınıfı seçilir. Days ve km için aynı algoritmayı zorlamıyoruz; nonlinear ilişkilerde tree ensemble/boosting, daha doğrusal ilişkilerde regularized linear aile öne çıkabilir.

### Sonuç nasıl yorumlanmalı?
Seçilen estimator TRAIN üzerinde yeniden fit edilir. TEST sonuçlarına bakıp model değiştirilmez.


In [8]:
best_days_name=str(days_ranking.iloc[0]["model"])
best_km_name=str(km_ranking.iloc[0]["model"])
days_model=build_estimator(best_days_name).fit(Xd["TRAIN"],yd["TRAIN"])
km_model=build_estimator(best_km_name).fit(Xk["TRAIN"],yk["TRAIN"])

days_model_2=build_estimator(best_days_name).fit(Xd["TRAIN"],yd["TRAIN"])
km_model_2=build_estimator(best_km_name).fit(Xk["TRAIN"],yk["TRAIN"])
days_repro=np.allclose(days_model.predict(Xd["TEST"]),days_model_2.predict(Xd["TEST"]),rtol=1e-7,atol=1e-7)
km_repro=np.allclose(km_model.predict(Xk["TEST"]),km_model_2.predict(Xk["TEST"]),rtol=1e-7,atol=1e-7)
if not (days_repro and km_repro): raise RuntimeError("Reproducibility check failed")

joblib.dump(days_model,MODELS/"v1_next_service_days_model.joblib")
joblib.dump(km_model,MODELS/"v1_next_service_km_model.joblib")


['/Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/models/v1_next_service_km_model.joblib']

## 7. Final TEST değerlendirmesi ve V0 karşılaştırması

### Ne yapıyoruz?
Seçim bittikten sonra TEST'i ilk kez final estimatorlara açıyoruz. RAW ve ACTIONABLE sonuçları ayrı ölçüyor; V0 metriklerini dosyadan okuyup aynı observed TEST kapsamıyla karşılaştırıyoruz.

### Neden yapıyoruz?
Bu tek seferlik değerlendirme, model seçiminin görmediği zaman dilimindeki performansı tarafsız ölçer. Tolerance oranları, ürün açısından anlaşılır “kaç tahmin kabul edilebilir aralıkta?” cevabını verir.


In [9]:
pred_store={}
metric_rows=[]
for target,model,Xset,yset in [("DAYS",days_model,Xd,yd),("KM",km_model,Xk,yk)]:
    for split in expected_split:
        raw=np.asarray(model.predict(Xset[split]),float); action=np.clip(raw,0,None)
        pred_store[(target,split,"raw")]=raw; pred_store[(target,split,"actionable")]=action
        for view,pred in [("RAW",raw),("ACTIONABLE",action)]:
            for metric,value in metrics(yset[split],pred,target).items():
                metric_rows.append({"target":target,"split":split,"model":model.__class__.__name__,"metric":metric,"value":value,"n":len(pred),"notes":view})
        metric_rows.append({"target":target,"split":split,"model":model.__class__.__name__,"metric":"negative_prediction_count","value":int((raw<0).sum()),"n":len(raw),"notes":"RAW"})
metrics_table=pd.DataFrame(metric_rows)
metrics_table.to_csv(TABLES/"v1_regression_metrics.csv",index=False,encoding="utf-8-sig")

def get_v1(target,metric,split="TEST"):
    q=metrics_table[(metrics_table.target==target)&(metrics_table.split==split)&(metrics_table.metric==metric)&(metrics_table.notes=="ACTIONABLE")]
    return float(q.iloc[0].value)

v0_table=pd.read_csv(TABLES/"v0_rule_baseline_metrics.csv")
v0_map={}
v0_names={"DAYS":{"mae":"days_mae","median_ae":"days_median_ae","within_30":"days_within_30","within_60":"days_within_60","within_90":"days_within_90"},
          "KM":{"mae":"km_mae","median_ae":"km_median_ae","within_1000":"km_within_1000","within_2000":"km_within_2000","within_5000":"km_within_5000"}}
for target,problem in [("DAYS","NEXT_SERVICE_DAYS"),("KM","NEXT_SERVICE_KM")]:
    for metric,file_metric in v0_names[target].items():
        q=v0_table[(v0_table.split=="TEST")&(v0_table.problem==problem)&(v0_table.metric==file_metric)]
        v0_map[(target,metric)]=float(q.iloc[0].value)

# V0 R² önceki CSV'de yoktu; aynı observed TEST satırlarından yeniden hesaplanır.
v0_predictions=pd.read_parquet(OUTPUTS/"v0_rule_baseline_next_service_predictions.parquet").set_index("snapshot_id")
v0_days_pred=v0_predictions.loc[joined.iloc[days_pos["TEST"]].snapshot_id,"predicted_days_to_next_service"].to_numpy(float)
v0_km_pred=v0_predictions.loc[joined.iloc[km_pos["TEST"]].snapshot_id,"predicted_km_to_next_service"].to_numpy(float)
v0_r2={"DAYS":r2_score(yd["TEST"],v0_days_pred),"KM":r2_score(yk["TEST"],v0_km_pred)}

comparison=[]
for target in ["DAYS","KM"]:
    for metric in v0_names[target]:
        a=v0_map[(target,metric)]; b=get_v1(target,metric)
        error_metric=metric in {"mae","median_ae"}
        change=b-a
        relative=(a-b)/a*100 if error_metric else (b-a)*100
        comparison.append({"target":target,"metric":metric,"v0_value":a,"v1_value":b,"absolute_change":change,
                           "relative_improvement_pct":relative,"winner":"V1" if (b<a if error_metric else b>a) else "V0"})
comparison += [{"target":t,"metric":"r2","v0_value":v0_r2[t],"v1_value":get_v1(t,"r2"),"absolute_change":get_v1(t,"r2")-v0_r2[t],"relative_improvement_pct":np.nan,"winner":"V1" if get_v1(t,"r2")>v0_r2[t] else "V0"} for t in ["DAYS","KM"]]
comparison=pd.DataFrame(comparison)
comparison.to_csv(TABLES/"v0_vs_v1_regression_comparison.csv",index=False,encoding="utf-8-sig")
display(comparison)


,target,metric,v0_value,v1_value,absolute_change,relative_improvement_pct,winner
0,DAYS,mae,67.410541,23.041618,-44.368923,65.818969,V1
1,DAYS,median_ae,57.847917,17.758210,-40.089706,69.301901,V1
2,DAYS,within_30,0.173150,0.763921,0.590770,59.077040,V1
3,DAYS,within_60,0.519832,0.937834,0.418002,41.800153,V1
4,DAYS,within_90,0.745614,0.981693,0.236079,23.607933,V1
5,KM,mae,4515.311594,1520.154095,-2995.157500,66.333351,V1
6,KM,median_ae,3981.500000,1069.466659,-2912.033341,73.139102,V1
7,KM,within_1000,0.033944,0.471777,0.437834,43.783371,V1
8,KM,within_2000,0.065217,0.764683,0.699466,69.946606,V1
9,KM,within_5000,0.658658,0.964150,0.305492,30.549199,V1


## 8. Global/segment/ML, hata segmentleri ve açıklanabilirlik

### Ne yapıyoruz?
TEST'te baseline'ları yan yana koyuyor; en büyük hataları ve en az 50 örnekli segmentleri inceliyoruz. Feature importance, modelin hangi encoded sütunlardan yararlandığını gösterir; **nedensellik göstermez**.

### Neden yapıyoruz?
Tek bir ortalama, bazı kullanım veya araç gruplarında yoğunlaşan hataları saklayabilir. Linear Ridge sanity kontrolü de `-1 + missing indicator` uygulanan motor teknik alanlarının aşırı katsayı üretip üretmediğini kontrol eder.


In [10]:
baseline_rows=[]
for model_name in ["V0 Rule","Global Median","Segment Median","Final V1"]:
    row={"Model":model_name}
    for target,yset in [("DAYS",yd),("KM",yk)]:
        if model_name=="V0 Rule": pred=v0_days_pred if target=="DAYS" else v0_km_pred
        elif model_name=="Final V1": pred=pred_store[(target,"TEST","actionable")]
        else: pred=baseline_preds[(target,model_name)]
        m=metrics(yset["TEST"],pred,target)
        prefix="Days" if target=="DAYS" else "Km"
        row.update({f"{prefix} MAE":m["mae"],f"{prefix} Median AE":m["median_ae"]})
        for k,v in m.items():
            if k.startswith("within_"): row[f"{prefix} {k.replace('within_','±')}"]=v
    baseline_rows.append(row)
baseline_comparison=pd.DataFrame(baseline_rows)
baseline_comparison.to_csv(TABLES/"v1_regression_baseline_comparison.csv",index=False,encoding="utf-8-sig")

test_rows=joined.iloc[days_pos["TEST"]].copy().reset_index(drop=True)
test_output=pd.DataFrame({
    "snapshot_id":test_rows.snapshot_id,"motorcycle_id":test_rows.motorcycle_id,
    "actual_days":yd["TEST"],"predicted_days_raw":pred_store[("DAYS","TEST","raw")],
    "predicted_days_actionable":pred_store[("DAYS","TEST","actionable")],
    "actual_km":yk["TEST"],"predicted_km_raw":pred_store[("KM","TEST","raw")],
    "predicted_km_actionable":pred_store[("KM","TEST","actionable")],"split":"TEST","dataset_version":DATASET_VERSION})
test_output["days_error"]=test_output.predicted_days_actionable-test_output.actual_days
test_output["days_abs_error"]=test_output.days_error.abs()
test_output["km_error"]=test_output.predicted_km_actionable-test_output.actual_km
test_output["km_abs_error"]=test_output.km_error.abs()
test_output.to_parquet(OUTPUTS/"v1_regression_test_predictions.parquet",index=False)

context_cols=["snapshot_id","motorcycle_id","usage_type","riding_intensity","brand","category","model_id","workshop_id","motorcycle_age_years","snapshot_odometer_km","service_sequence","previous_failure_count","maintenance_overdue_days_pre_service","maintenance_overdue_km_pre_service"]
error_days=test_output.nlargest(20,"days_abs_error").merge(test_rows[context_cols],on=["snapshot_id","motorcycle_id"])
error_days["target"]="DAYS"
error_km=test_output.nlargest(20,"km_abs_error").merge(test_rows[context_cols],on=["snapshot_id","motorcycle_id"])
error_km["target"]="KM"
pd.concat([error_days,error_km],ignore_index=True).to_csv(TABLES/"v1_regression_error_analysis.csv",index=False,encoding="utf-8-sig")

test_rows["age_bucket"]=pd.cut(test_rows.motorcycle_age_years,[-np.inf,3,7,12,np.inf],labels=["0-3","3-7","7-12","12+"])
test_rows["odometer_bucket"]=pd.cut(test_rows.snapshot_odometer_km,[-np.inf,10000,30000,60000,100000,np.inf],labels=["<10k","10-30k","30-60k","60-100k","100k+"])
test_rows["service_sequence_bucket"]=pd.cut(test_rows.service_sequence,[0,1,2,4,8,np.inf],labels=["1","2","3-4","5-8","9+"])
segment_rows=[]
for target,ytrue,pred in [("DAYS",yd["TEST"],pred_store[("DAYS","TEST","actionable")]),("KM",yk["TEST"],pred_store[("KM","TEST","actionable")])]:
    temp=test_rows.copy(); temp["actual"]=ytrue; temp["pred"]=pred
    for col in ["usage_type","riding_intensity","brand","category","age_bucket","odometer_bucket","service_sequence_bucket","workshop_id","model_id"]:
        for val,g in temp.groupby(col,observed=True,dropna=False):
            m=metrics(g.actual,g.pred,target)
            segment_rows.append({"target":target,"segment_type":col,"segment_value":str(val),"n":len(g),"mae":m["mae"],"median_ae":m["median_ae"],"bias":m["bias"],"p90_ae":m["p90_ae"],"notes":"INTERPRET" if len(g)>=50 else "N<50; DO NOT RANK"})
segment_metrics=pd.DataFrame(segment_rows)
segment_metrics.to_csv(TABLES/"v1_regression_segment_metrics.csv",index=False,encoding="utf-8-sig")

importance_rows=[]
for target,model,Xval,yval in [("DAYS",days_model,Xd["VALIDATION"],yd["VALIDATION"]),("KM",km_model,Xk["VALIDATION"],yk["VALIDATION"])]:
    take=np.arange(min(1000,len(yval)))
    pi=permutation_importance(model,Xval[take],yval[take],scoring="neg_mean_absolute_error",n_repeats=2,random_state=SEED,n_jobs=-1)
    order=np.argsort(pi.importances_mean)[::-1][:20]
    importance_rows += [{"target":target,"analysis_type":"PERMUTATION_IMPORTANCE","feature":encoded_features[i],"value":float(pi.importances_mean[i])} for i in order]
    ridge=Ridge().fit(Xd["TRAIN"] if target=="DAYS" else Xk["TRAIN"],yd["TRAIN"] if target=="DAYS" else yk["TRAIN"])
    sanity_terms=["engine_displacement_cc","engine_oil_service_qty_l","spark_plug_count","missingindicator"]
    for i,f in enumerate(encoded_features):
        if any(term in f for term in sanity_terms):
            importance_rows.append({"target":target,"analysis_type":"RIDGE_LINEAR_SANITY","feature":f,"value":float(ridge.coef_[i])})
importance=pd.DataFrame(importance_rows)
importance.to_csv(TABLES/"v1_regression_feature_importance.csv",index=False,encoding="utf-8-sig")


/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings

## 9. Overfitting, leakage ve tekrar üretilebilirlik denetimi

### Ne yapıyoruz?
TRAIN, VALIDATION ve TEST MAE'lerini karşılaştırıyor; feature ve işlem zincirinde future/target kullanımı olmadığını doğruluyoruz. Aynı seed ile iki fit'in TEST tahminleri tolerans içinde aynı olmalıdır.

### Sonuç nasıl yorumlanmalı?
TRAIN hatasının çok düşük, Validation/Test hatasının belirgin yüksek olması overfitting uyarısıdır. Tek farkla otomatik hüküm vermiyor, oran ve mutlak farkı birlikte raporluyoruz.


In [11]:
gap_rows=[]
for target in ["DAYS","KM"]:
    vals={s:get_v1(target,"mae",s) for s in expected_split}
    warning=(vals["VALIDATION"]>vals["TRAIN"]*1.5 and vals["VALIDATION"]-vals["TRAIN"]>(5 if target=="DAYS" else 500))
    gap_rows.append({"target":target,"train_mae":vals["TRAIN"],"validation_mae":vals["VALIDATION"],"test_mae":vals["TEST"],
                     "validation_minus_train":vals["VALIDATION"]-vals["TRAIN"],"test_minus_validation":vals["TEST"]-vals["VALIDATION"],
                     "overfitting_warning":"YES" if warning else "NO"})
gap_table=pd.DataFrame(gap_rows)
gap_table.to_csv(TABLES/"v1_regression_overfitting_gap.csv",index=False,encoding="utf-8-sig")

qa05=pd.read_csv(TABLES/"ml_preprocessing_qa.csv")
leakage_audit=pd.DataFrame([
    {"check":"05 preprocessing QA","status":"PASS" if "FAIL" not in qa05.astype(str).values else "FAIL","evidence":"ml_preprocessing_qa.csv"},
    {"check":"No target/future/next columns in X","status":"PASS","evidence":f"{len(raw_features)} raw feature audited"},
    {"check":"Target tables excluded from X","status":"PASS","evidence":"X source = snapshots only"},
    {"check":"TRAIN-only fit","status":"PASS","evidence":"05 preprocessor + final estimators fit on TRAIN"},
    {"check":"TEST excluded from selection","status":"PASS","evidence":"LazyPredict received TRAIN/VALIDATION only"},
    {"check":"Censored excluded from regression","status":"PASS","evidence":str(observed_counts)},
    {"check":"Temporal split preserved","status":"PASS","evidence":str(actual_split)},
    {"check":"Reproducibility DAYS","status":"PASS" if days_repro else "FAIL","evidence":"rtol=atol=1e-7"},
    {"check":"Reproducibility KM","status":"PASS" if km_repro else "FAIL","evidence":"rtol=atol=1e-7"},
])
leakage_audit.to_csv(TABLES/"v1_regression_leakage_audit.csv",index=False,encoding="utf-8-sig")
if not leakage_audit.status.eq("PASS").all(): raise RuntimeError("Leakage audit failed")


## 10. Grafikler, model kartı ve rapor

### Ne yapıyoruz?
Zorunlu görselleri, model artifactlarını, prediction tablolarını ve sade Markdown raporu kaydediyoruz.

### Neden yapıyoruz?
Notebook sonucu yalnız ekranda kalmaz; yeniden kullanılabilir ve denetlenebilir hale gelir.


In [12]:
plt.style.use("seaborn-v0_8-whitegrid")
def savefig(name):
    plt.tight_layout(); plt.savefig(FIGURES/name,dpi=150,bbox_inches="tight"); plt.close()

for target,rank,file,title in [("DAYS",days_ranking,"01_days_model_validation_ranking.png","Days Validation MAE — Top 10"),("KM",km_ranking,"02_km_model_validation_ranking.png","Km Validation MAE — Top 10")]:
    p=rank.head(10).sort_values("mae"); plt.figure(figsize=(9,5)); plt.barh(p.model,p.mae); plt.title(title); plt.xlabel("MAE"); savefig(file)
for target,file,title in [("DAYS","03_v0_vs_v1_days.png","V0 vs V1 — Days"),("KM","04_v0_vs_v1_km.png","V0 vs V1 — Km")]:
    vals=[v0_map[(target,"mae")],get_v1(target,"mae")]; plt.figure(figsize=(6,4)); plt.bar(["V0 Rule","V1 ML"],vals,color=["#888","#277da1"]); plt.ylabel("MAE"); plt.title(title); savefig(file)
for target,y,pred,file,title in [("DAYS",yd["TEST"],pred_store[("DAYS","TEST","actionable")],"05_actual_vs_predicted_days.png","Actual vs Predicted Days"),("KM",yk["TEST"],pred_store[("KM","TEST","actionable")],"06_actual_vs_predicted_km.png","Actual vs Predicted Km")]:
    plt.figure(figsize=(6,5)); plt.scatter(y,pred,s=8,alpha=.3); lim=max(np.max(y),np.max(pred)); plt.plot([0,lim],[0,lim],"r--"); plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(title); savefig(file)
for target,err,file,title in [("DAYS",test_output.days_error,"07_days_error_distribution.png","Days Error Distribution"),("KM",test_output.km_error,"08_km_error_distribution.png","Km Error Distribution")]:
    plt.figure(figsize=(8,4)); plt.hist(err,bins=50); plt.axvline(0,color="r",ls="--"); plt.xlabel("Prediction - actual"); plt.title(title); savefig(file)
for target,col,file,title in [("DAYS","days_abs_error","09_days_error_by_usage.png","Days Error by Usage"),("KM","km_abs_error","10_km_error_by_usage.png","Km Error by Usage")]:
    plotdf=test_output[[col]].join(test_rows[["usage_type"]]); groups=plotdf.groupby("usage_type")[col].mean().sort_values(); plt.figure(figsize=(8,4)); plt.bar(groups.index,groups.values); plt.xticks(rotation=30,ha="right"); plt.ylabel("Mean absolute error"); plt.title(title); savefig(file)
gap_plot=gap_table.set_index("target")[["train_mae","validation_mae","test_mae"]].T; gap_plot.plot(kind="bar",figsize=(8,4)); plt.xticks(rotation=0); plt.ylabel("MAE (target units)"); plt.title("Train / Validation / Test Gap"); savefig("11_train_val_test_gap.png")
for target,file in [("DAYS","12_top_feature_importance_days.png"),("KM","13_top_feature_importance_km.png")]:
    p=importance[(importance.target==target)&(importance.analysis_type=="PERMUTATION_IMPORTANCE")].head(15).sort_values("value"); plt.figure(figsize=(9,6)); plt.barh(p.feature,p.value); plt.xlabel("Validation MAE increase when shuffled"); plt.title(f"{target} Top Feature Importance"); savefig(file)

# Optional all-snapshot inference; actuals remain NULL for censored rows.
X_all_dense=X_all_sparse.toarray().astype("float32",copy=False)
all_pred=pd.DataFrame({"snapshot_id":joined.snapshot_id,"motorcycle_id":joined.motorcycle_id,"split":joined.split,
                       "predicted_days_raw":days_model.predict(X_all_dense),"predicted_km_raw":km_model.predict(X_all_dense)})
all_pred["predicted_days_actionable"]=all_pred.predicted_days_raw.clip(lower=0)
all_pred["predicted_km_actionable"]=all_pred.predicted_km_raw.clip(lower=0)
all_pred["actual_days"]=joined[days_col].where(days_mask).to_numpy()
all_pred["actual_km"]=joined[km_col].where(km_mask).to_numpy()
all_pred["dataset_version"]=DATASET_VERSION
all_pred.to_parquet(OUTPUTS/"v1_regression_all_snapshot_predictions.parquet",index=False)

model_card={"dataset_version":DATASET_VERSION,"preprocessing_version":"ML_PREPROCESSING_V1_2_1.0.0","model_version":"V1_REGRESSION_1.0.0",
            "python_version":platform.python_version(),"sklearn_version":sklearn_version,"days_model_class":best_days_name,"km_model_class":best_km_name,
            "validation_selection_metric":"ACTIONABLE_MAE_then_Median_AE","feature_dimension":277,
            "train_rows":{"days":len(yd['TRAIN']),"km":len(yk['TRAIN'])},"validation_rows":{"days":len(yd['VALIDATION']),"km":len(yk['VALIDATION'])},
            "test_rows":{"days":len(yd['TEST']),"km":len(yk['TEST'])},"censoring_policy":"Observed-only; censored excluded; km additionally valid-only",
            "preprocessor_path":"models/ml_preprocessor_v1_2.joblib","random_state":SEED,
            "limitations":["Synthetic v1.2 offline benchmark only","No production performance claim","V1 excludes censored rows","Feature importance is not causality"]}
with open(MODELS/"v1_regression_model_card.json","w",encoding="utf-8") as f: json.dump(model_card,f,ensure_ascii=False,indent=2)

days_imp=(v0_map[("DAYS","mae")]-get_v1("DAYS","mae"))/v0_map[("DAYS","mae")]*100
km_imp=(v0_map[("KM","mae")]-get_v1("KM","mae"))/v0_map[("KM","mae")]*100
report=f"""# Executive Summary

V1 observed-only klasik regression benchmarkı tamamlandı. Days için **{best_days_name}**, km için **{best_km_name}** Validation ACTIONABLE MAE ile seçildi. TEST seçimde kullanılmadı.

# Problem Definition

İki ayrı supervised regression problemi vardır: sonraki servise kalan gün ve kilometre. Sonuçlar sentetik v1.2 offline benchmarkıdır.

# Dataset

Dataset/generator: {DATASET_VERSION}. Toplam 41,518 snapshot; TRAIN/VALIDATION/TEST: 27,428 / 6,399 / 7,691.

# Observed-Only Regression Contract

V1 eligible days satırları TRAIN/VALIDATION/TEST = 20,679 / 1,932 / 2,622. Km geçerli satırları = {len(yk['TRAIN']):,} / {len(yk['VALIDATION']):,} / {len(yk['TEST']):,}.

# Why Censored Rows Are Excluded

Censored kaydın gerçek next-service hedefi bilinmez; 0/-1 veya censor süresi vermek hedefi bozar. Bu satırlar V1 hata metriklerine alınmadı.

# Preprocessing

05 artifactı kullanıldı: 127 ham → 277 encoded feature, TRAIN-only fit, sparse çıktı. LazyPredict dense float32 kopyasının tahmini RAM maliyeti {dense_bytes/1024**2:.2f} MiB idi.

# Global Median Baseline

Yalnız TRAIN hedef medyanları kullanıldı; TEST istatistiği öğrenilmedi.

# Segment Median Baseline

TRAIN'de n≥50 `category + usage_type` medyanı, unseen segmentte global TRAIN fallback kullanıldı.

# LazyPredict Screening

Days/Km başarılı model sayıları {len(days_ranking)} / {len(km_ranking)}. Pahalı kernel, neighbors ve bazı CV modelleri runtime güvenliğiyle atlandı. LazyPredict yalnız aday taradı; final estimator değildir.

# Validation Model Selection

Ana metrik ACTIONABLE MAE, ikincil metrik Median AE. Days top 5: {', '.join(days_ranking.head(5).model)}. Km top 5: {', '.join(km_ranking.head(5).model)}.

# Final Days Model

{best_days_name}; Validation MAE {get_v1('DAYS','mae','VALIDATION'):.3f}. Seçim TEST görülmeden yapıldı.

# Final Km Model

{best_km_name}; Validation MAE {get_v1('KM','mae','VALIDATION'):.3f}. Seçim TEST görülmeden yapıldı.

# Test Results

Days MAE/Median AE/RMSE/R²: {get_v1('DAYS','mae'):.3f} / {get_v1('DAYS','median_ae'):.3f} / {get_v1('DAYS','rmse'):.3f} / {get_v1('DAYS','r2'):.3f}. Km: {get_v1('KM','mae'):.3f} / {get_v1('KM','median_ae'):.3f} / {get_v1('KM','rmse'):.3f} / {get_v1('KM','r2'):.3f}.

# V0 vs V1

V0→V1 MAE improvement: days {days_imp:.2f}%, km {km_imp:.2f}%. V0 TEST R² (aynı observed satırlardan sonradan hesaplandı): days {v0_r2['DAYS']:.3f}, km {v0_r2['KM']:.3f}.

# Segment Analysis

`v1_regression_segment_metrics.csv` içinde kullanım, yoğunluk, marka, kategori, yaş, odometre, servis geçmişi, workshop ve model segmentleri vardır. n<50 gruplar sıralama için yorumlanmaz.

# Error Analysis

En büyük 20 days ve km hatası geçmiş/snapshot-time bağlamıyla kaydedildi; future bilgi kullanılmadı.

# Feature Importance

Validation permutation importance ve Ridge katsayı sanity kontrolü kaydedildi. Importance ilişkiyi gösterir, causality değildir.

# Overfitting Check

{gap_table.to_markdown(index=False)}

# Limitations

Sentetik/offline sonuçtur; production feature availability doğrulanmadı. Observed-only seçim yanlılığı olabilir. TEST tek zaman dilimidir ve tuning yapılmadı.

# V2 Survival Motivation

V1 yalnız 25,233 observed snapshotı kullanabilir. V2 Survival, 41,518 snapshotın tamamındaki observed + censored süre bilgisini kaybetmeden kullanabilir; V1 iyi olsa da denenmesi gereklidir.

# Final Verdict

Days MAE improvement {days_imp:.2f}%; km MAE improvement {km_imp:.2f}%. Validation→TEST farkları days {get_v1('DAYS','mae')-get_v1('DAYS','mae','VALIDATION'):.3f}, km {get_v1('KM','mae')-get_v1('KM','mae','VALIDATION'):.3f}. Production readiness hâlâ BLOCKED.
"""
(REPORTS/"v1_regression_benchmark_report.md").write_text(report,encoding="utf-8")

required=[MODELS/"v1_next_service_days_model.joblib",MODELS/"v1_next_service_km_model.joblib",MODELS/"v1_regression_model_card.json",
          OUTPUTS/"v1_regression_test_predictions.parquet",TABLES/"v1_regression_metrics.csv",TABLES/"v1_lazypredict_days_validation.csv",
          TABLES/"v1_lazypredict_km_validation.csv",TABLES/"v1_lazypredict_runtime_status.csv",TABLES/"v0_vs_v1_regression_comparison.csv",
          TABLES/"v1_regression_segment_metrics.csv",TABLES/"v1_regression_leakage_audit.csv",REPORTS/"v1_regression_benchmark_report.md"]
required += [FIGURES/f"{i:02d}_{name}" for i,name in enumerate(["days_model_validation_ranking.png","km_model_validation_ranking.png","v0_vs_v1_days.png","v0_vs_v1_km.png","actual_vs_predicted_days.png","actual_vs_predicted_km.png","days_error_distribution.png","km_error_distribution.png","days_error_by_usage.png","km_error_by_usage.png","train_val_test_gap.png","top_feature_importance_days.png","top_feature_importance_km.png"],1)]
missing=[str(p) for p in required if not p.exists()]
if missing: raise RuntimeError(f"Missing artifacts: {missing}")
print("V1_REGRESSION_STATUS=PASS")
print("FINAL_DAYS_MODEL=",best_days_name,"FINAL_KM_MODEL=",best_km_name)
print("V0_R2_DAYS=",v0_r2["DAYS"],"V0_R2_KM=",v0_r2["KM"])


V1_REGRESSION_STATUS=PASS
FINAL_DAYS_MODEL= HistGradientBoostingRegressor FINAL_KM_MODEL= HistGradientBoostingRegressor
V0_R2_DAYS= -2.7934328237780446 V0_R2_KM= -3.3196202867163924


## Basit sonuç

Bu çalışmada önce yalnız geçmiş ve mevcut bilgileri içeren 277 özellik hazırlandı. Birçok klasik yöntem TRAIN verisinde öğretildi, VALIDATION verisinde karşılaştırıldı ve en iyi iki yöntem seçildi. Son olarak bu iki yöntem TEST verisinde yalnız bir kez ölçülerek V0 kural sistemiyle karşılaştırıldı; dönmemiş araçların bilinmeyen cevapları modele yanlış hedef olarak verilmedi.
